# MaiaFormer-355 — HF ↔ GGUF F32 numerical validation

**Goal:** compare the full `[sequence_length, 65536]` logit tensors produced by Hugging Face `LlamaForCausalLM` and the llama.cpp runtime (via `llama-cpp-python`). This is a numerical equivalence test, **not** a language-quality test.

**Requirements:** `/content/maiaformer-355-hf/` and `/content/maiaformer-355-f32.gguf` already exist. Run cells in order. This notebook does not change either model or tokenizer.

**Important:** The tokenizer in this exported checkpoint is temporary (GPT-2 extended to 65,536). Replace it before actual pretraining. This validation uses *explicit token IDs* to avoid differences in tokenization.


In [ ]:
from pathlib import Path
HF_DIR = Path('/content/maiaformer-355-hf')
GGUF_PATH = Path('/content/maiaformer-355-f32.gguf')
assert (HF_DIR / 'config.json').is_file(), f'Missing: {HF_DIR}/config.json'
assert (HF_DIR / 'model.safetensors').is_file(), f'Missing: {HF_DIR}/model.safetensors'
assert GGUF_PATH.is_file(), f'Missing: {GGUF_PATH}'
print('HF:', HF_DIR)
print('GGUF:', GGUF_PATH, f'({GGUF_PATH.stat().st_size / 2**20:.1f} MiB)')


## 1. Isolated environment

The Colab main Python environment was previously affected by conflicting dependency versions. Create a **new** virtualenv for this validation. The llama.cpp Python binding compiles a CPU runtime; it may take several minutes. Install Torch CPU to avoid a large CUDA wheel. The notebook invokes the isolated interpreter through subprocess, without modifying the Colab kernel.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'virtualenv'], check=True)
subprocess.run([sys.executable, '-m', 'virtualenv', '/content/maia-compare-env'], check=True)
PY = '/content/maia-compare-env/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', 'transformers>=4.46,<5', 'safetensors', 'numpy<3', 'sentencepiece'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', 'llama-cpp-python'], check=True)
print('Ready:', PY)


## 2. Full-logit comparison script

The C++ binding returns `scores` for all evaluated positions when `logits_all=True`. We disable BOS insertion and pass the same explicit IDs to both runtimes. We test short sequences to keep CPU execution manageable.

If the binding build fails, retain the error log and do not interpret it as an architecture failure. The C++ binding is a separate installation from the already-built `/content/llama.cpp` executable.


In [ ]:
script = r'''import json, math, time
import numpy as np
import torch
from pathlib import Path
from transformers import LlamaForCausalLM
from llama_cpp import Llama

hf_dir = '/content/maiaformer-355-hf'
gguf_path = '/content/maiaformer-355-f32.gguf'
torch.set_num_threads(4)
print('Loading HF model (CPU, FP32)...', flush=True)
hf = LlamaForCausalLM.from_pretrained(hf_dir, torch_dtype=torch.float32, low_cpu_mem_usage=False)
hf.eval()
config = hf.config
print('HF config:', {'vocab_size': config.vocab_size, 'hidden_size': config.hidden_size,
 'num_hidden_layers': config.num_hidden_layers, 'num_attention_heads': config.num_attention_heads,
 'num_key_value_heads': config.num_key_value_heads}, flush=True)
print('Loading GGUF model (CPU, F32)...', flush=True)
llm = Llama(model_path=gguf_path, n_ctx=128, n_batch=32, n_gpu_layers=0,
            logits_all=True, embedding=False, verbose=False)

# Same token IDs in both engines. Avoid special-token assumptions and tokenizer differences.
sequences = {
    'short': [15496, 11, 995, 0],
    'mixed': [42, 1337, 1024, 32000, 50256, 65000, 7, 256],
    'repeated': [101, 101, 101, 101, 101, 101, 101, 101],
}
results = []
for name, ids in sequences.items():
    assert all(0 <= t < config.vocab_size for t in ids), (name, ids)
    print('\nSequence:', name, ids, flush=True)
    with torch.inference_mode():
        logits_hf = hf(input_ids=torch.tensor([ids], dtype=torch.long),
                       use_cache=False).logits[0].cpu().numpy().astype(np.float64)
    llm.reset()
    llm.eval(ids)
    logits_gguf = np.asarray(llm.scores, dtype=np.float64)
    # llama-cpp-python versions can allocate additional rows; only the last len(ids)
    # are relevant after reset() and eval(ids).
    assert logits_gguf.shape[0] >= len(ids), logits_gguf.shape
    logits_gguf = logits_gguf[:len(ids), :]
    assert logits_hf.shape == logits_gguf.shape, (logits_hf.shape, logits_gguf.shape)
    assert np.isfinite(logits_hf).all() and np.isfinite(logits_gguf).all(), 'Non-finite logits'
    diff = np.abs(logits_hf - logits_gguf)
    rel = diff / np.maximum(np.abs(logits_hf), 1e-3)
    top_hf = np.argmax(logits_hf, axis=-1)
    top_gguf = np.argmax(logits_gguf, axis=-1)
    # Stable softmax and KL divergence (HF || GGUF)
    def logsoftmax(x):
        z = x - x.max(axis=-1, keepdims=True)
        return z - np.log(np.exp(z).sum(axis=-1, keepdims=True))
    logp, logq = logsoftmax(logits_hf), logsoftmax(logits_gguf)
    kl = (np.exp(logp) * (logp - logq)).sum(axis=-1)
    row = {
      'sequence': name, 'positions': len(ids), 'vocab_size': logits_hf.shape[-1],
      'max_abs_error': float(diff.max()), 'mean_abs_error': float(diff.mean()),
      'p99_abs_error': float(np.percentile(diff, 99)),
      'max_relative_error_floor_1e-3': float(rel.max()),
      'mean_relative_error_floor_1e-3': float(rel.mean()),
      'argmax_agreement': float(np.mean(top_hf == top_gguf)),
      'hf_argmax': top_hf.tolist(), 'gguf_argmax': top_gguf.tolist(),
      'max_kl_hf_to_gguf_nats': float(np.max(np.maximum(kl, 0))),
      'mean_kl_hf_to_gguf_nats': float(np.mean(np.maximum(kl, 0))),
    }
    results.append(row)
    print(json.dumps(row, indent=2), flush=True)

out = {'model': 'MaiaFormer-355', 'format': 'GGUF F32', 'results': results}
Path('/content/maiaformer-hf-gguf-validation.json').write_text(json.dumps(out, indent=2))
print('\nSaved /content/maiaformer-hf-gguf-validation.json', flush=True)
'''
from pathlib import Path
Path('/content/compare_maiaformer_hf_gguf.py').write_text(script, encoding='utf-8')
print('Script created.')


In [ ]:
import subprocess
with open('/content/maiaformer-hf-gguf-validation.log', 'w') as log:
    p = subprocess.run([PY, '-u', '/content/compare_maiaformer_hf_gguf.py'],
                       stdout=log, stderr=subprocess.STDOUT, text=True)
print('Exit code:', p.returncode)
print(Path('/content/maiaformer-hf-gguf-validation.log').read_text()[-18000:])
if p.returncode != 0:
    print('Comparison did not finish. Share the log for diagnosis; no equivalence conclusion yet.')


## 3. How to interpret results

- **`argmax_agreement = 1.0`** for all positions is encouraging but **not sufficient** on its own.
- Examine **max/mean absolute errors**, the distribution of errors and **KL divergence**. Differences need not be zero because PyTorch and ggml use different CPU kernels and accumulation orders.
- If errors are large or the top token differs, investigate **tensor mapping, RoPE, RMSNorm, attention masking, tied embeddings and tokenizer metadata**. A single failing test does not identify which component is wrong.
- The short sequences test causal prefill. For complete parity, a subsequent test should compare **cached incremental decoding** as well as prefill.
- The Python binding may use a different llama.cpp commit than the separately compiled CLI. Record the versions before claiming parity for a specific commit.

**Share the output of the last cell**. We can then establish measured tolerances and decide whether the implementation passes.
